# Region-aware RNA–protein fusion

Does knowing where an RNA transcript encodes protein make fusion more useful? This notebook prepares Mina’s **Step A.6**: human mRNA half-life first, then human translation efficiency. It compares RNA alone with zero-filled protein features, an explicit coding-region mask, position-wise gates and gated cross-attention. This is a new controlled adaptation on frozen encoders; it is not a published IsoFormer or BioLangFusion reproduction. [Mina’s plan, Section A, Step 6](../../docs/Biological_Fusion_Plan.docx)

The inputs are complete mature transcripts from the local mRNABench tables. Half-life and translation efficiency remain separate targets: persistence does not equal translation rate. See the [target guide](../../docs/methods/data-and-targets.md) and the earlier [pooled half-life/TE investigation](halflife_translation_efficiency.ipynb). Gantt/Details **task 42** supplies the earlier region-aware research route.

The default run reads saved JSON evidence and plots saved summaries. It cannot prepare a cohort, extract embeddings or train. There are no new research results in this notebook yet. The preparation, extraction, training, diagnostics and saved-prediction analysis switches are separate.

Mina’s preceding experiments ask whether useful contributions vary along a sequence and whether selective auxiliary updates help preserve a strong anchor. A.6 asks those questions where the sequence regions have known biological meanings. Those preceding experiments are not assumed to have succeeded. This notebook first inspects whether our cohort, context windows and control masks can support that question, then prepares and compares the predictors.


---

## Setup

Use the [repository environment setup](../../README.md#setup). Dataset parquet files and the human homology map must already be present locally; the reader does not download them. Token extraction is the only stage that may load pretrained encoder weights, and it is disabled by default.


In [ ]:
from dataclasses import asdict, replace
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import torch


In [ ]:
# Resolve the checkout from a repository, Code, or notebook working directory.
start = Path.cwd().resolve()
REPO_DIR = next((path for path in (start, *start.parents)
                 if (path / "Code" / "shared_code").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Open this notebook from the repository or one of its subdirectories.")
CODE_DIR = REPO_DIR / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code.fusion import region_experiment as region
from shared_code.fusion import region_training as study
from shared_code.fusion import run_identity, train


### Settings and execution boundaries

The two RNA encoders are alternatives, each paired with ESM-2 8M. Native RNA and region-aware triplet pooling are both tested. The eight core arms per encoder/grid produce **32 fits per seed and dataset**, or **192 planned fits** over both datasets and training seeds 0, 1 and 2. This is a panel size, not an execution-progress count. The optional ungated-attention and nongated positional controls add 72 fits if all are requested; they are disabled.

| Setting | Basis |
| --- | --- |
| Human half-life, then TE; RNA/protein correspondence; A–E and mask controls | Mina’s Step A.6 |
| Mapped homology groups plus unmapped-gene groups; 70/15/15 allocation, split seed 42 | Agreed project evaluation policy; local homology map required |
| Training seeds 0, 1, 2; separate mask/bootstrap seed 0 | Project reproducibility choices, not biological constants |
| Terminal stop remains in CDS but supplies no amino acid; invalid/ambiguous input rows excluded from every arm | Agreed common-cohort policy |
| Complete non-overlapping encoder windows, reserving two structural-token slots | Explicit context adaptation; preserves coverage, not interactions between windows |
| 600-dimensional projections, joint affine sigmoid gates, four attention heads, dropout 0.1; 100-channel TextCNN | Retained project baseline dimensions, not parameters specified in Mina’s document |
| Adam/training budget below; native versus pooled grid | Retained optimizer policy and an explicit new comparison; equal kernel widths span different numbers of bases |

The map, window choices and any result-producing action are recorded with the actual inputs. Hardware selection does not change the cohort or model settings. Full native-position attention may be expensive: profile a small authorized job before scheduling the panel; no runtime or peak-memory estimate is established here.

The `scorer_width` field is retained for interface compatibility; the joint affine gates do not use a hidden scoring layer. The original training budget is a retained project baseline, not a budget estimated for full transcripts. The settings below do not authorize all prepared fits at once.


### Where the decisions come from

[Mina's A.6 proposal](../../docs/Biological_Fusion_Plan.docx) supplies the biological question and A-E comparison. The remaining rules make it executable on our data; their presence in code does not establish their scientific adequacy.

| Choice | Basis and consequence | What would warrant review? |
| --- | --- | --- |
| Native RNA positions; protein only at corresponding coding residues | Mina A.6; standard-code translation and terminal-stop masking are agreed project rules | Exclusions concentrate in a particular length or region population |
| Required homology map plus gene identity for unmapped rows; split seed 42 | Agreed project grouping; unknown homologs remain possible | Too few independent groups, concentrated groups or remaining exact overlap |
| Independent 1,022-position encoder windows | Project context adaptation using the current encoder limits and two structural-token slots | Frequent fragmentation of relevant regions or an impractical workload |
| Within-partition relative-coordinate mask permutation | Agreed reconstruction of Mina's shuffled-region control | Different donor masks leave most effective protein updates unchanged |
| D: joint RNA/protein affine gates, then C's concatenation mixer and mask feature | Gate inputs follow A.4; retaining C's readout is our A.6 adaptation, rather than A.4's literal weighted-sum output | The claimed comparison depends on changing the readout as well as gating |
| E: joint anchor/update affine gate, RNA skip, per-update LayerNorm and attention dropout | Gate follows A.5; normalization, projection and dropout choices are inherited project settings | Update scale or optimization explains a gain attributed to routing |
| Optional `attention_uniform` method key | Legacy name for learned attention with unit update weights and unrestricted RNA queries; it is not uniform token attention | A claim that this isolates gating while holding CDS query availability fixed |

The optional nongated positional mixer is also a project design. Parameter counts alone do not establish matched expressivity. Review the control's bottleneck and the measured resource trial before selecting the Newton workload; no audit automatically changes settings or adds training arms.


In [ ]:
DATASET_KEY = "rnahl-human"  # Follow with "translation-efficiency-human" after half-life review.
SEED = 0
TRAINING_SEEDS = (0, 1, 2)
SPLIT_SEED = 42
UNMAPPED_GENE_POLICY = "gene"  # Keep unmapped rows together by their recorded gene identifier.
MASK_SEED = 0
BOOTSTRAP_SEED = 0
BOOTSTRAP_DRAWS = 2000

READ_SAVED_RESULTS = True
RUN_INPUT_AUDIT = False  # Inspect the agreed cohort/control; do not publish a training split.
RUN_PREPARATION = False
RUN_EXTRACTION = False
RUN_TRAINING = False
RUN_ANALYSIS = False
RUN_DIAGNOSTICS = False
INCLUDE_OPTIONAL = False
FIT_KEYS = None  # None selects the declared core panel; a list permits a bounded smoke batch.

DATA_CACHE_ROOT = REPO_DIR / ".data-cache" / "mrnabench"
TOKEN_CACHE_ROOT = REPO_DIR / ".data-cache" / "region_tokens_v1"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "region_aware_v1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
PRODUCER = None  # Newton supplies job evidence; a requested local input audit records its CPU host.

# Sizes are content positions (bases or residues), not total model tokens.
WINDOW_LENGTHS = {"rnafm": 1022, "rinalmo": 1022, "esm2_8m": 1022}
CACHE_SHARD_ROWS = 256
CACHE_DTYPE = "float16"
AUDIT_WIDTHS = {"rnafm": 640, "rinalmo": 640, "esm2_8m": 320}  # Checkpoint feature widths; payload only.
BASE_CONFIG = train.TrainConfig(
    lr=3e-5, weight_decay=1e-5, batch_size=32, max_epochs=500,
    early_stopping_patience=20, plateau_patience=5, entropy_lambda=0.,
    channels=100, tconv="equation", seed=0,
)
CONFIG = replace(BASE_CONFIG, seed=SEED)
MODEL_SETTINGS = {
    "projection_width": 600, "scorer_width": 100, "n_heads": 4, "dropout": 0.1,
    "projection_activation": "tanh", "projection_norm": "none", "update_norm": "layer",
    "control_bottleneck": 100,  # Optional affine control; count and bottleneck both need comparison.
}


In [ ]:
actions = [RUN_INPUT_AUDIT, RUN_PREPARATION, RUN_EXTRACTION, RUN_TRAINING, RUN_ANALYSIS, RUN_DIAGNOSTICS]
if sum(actions) > 1:
    raise ValueError("Choose one result-producing action per execution.")
if READ_SAVED_RESULTS and (RUN_PREPARATION or RUN_EXTRACTION or RUN_TRAINING):
    raise ValueError("Disable READ_SAVED_RESULTS explicitly for preparation, extraction or training.")
if RUN_INPUT_AUDIT and not READ_SAVED_RESULTS:
    raise ValueError("Input inspection uses READ_SAVED_RESULTS=True with other actions disabled.")
if DATASET_KEY not in region.DATASET_KEYS or SEED not in TRAINING_SEEDS:
    raise ValueError("Choose one declared human dataset and training seed.")
if any(actions[1:]) and not PRODUCER:
    raise ValueError("An explicit action needs observed producer metadata from the approved launcher.")

STUDY_ROOT = Path(RESULTS_ROOT) / DATASET_KEY
PREPARATION_PATH = STUDY_ROOT / "preparation.json"
INPUT_AUDIT_PATH = STUDY_ROOT / "input-audit-v1.json"
PANEL = study.fit_panel(include_optional=INCLUDE_OPTIONAL)
if FIT_KEYS is not None:
    unknown = set(FIT_KEYS) - {spec.key for spec in PANEL}
    if unknown:
        raise ValueError(f"Unknown fit keys: {sorted(unknown)}")
    PANEL = tuple(spec for spec in PANEL if spec.key in FIT_KEYS)
    if not PANEL:
        raise ValueError("The selected fit panel is empty.")
print(f"Dataset: {DATASET_KEY}; predictor seed: {SEED}; device: {DEVICE}")
print(f"Selected panel: {len(PANEL)} fits; saved results: {STUDY_ROOT}")


---

## Read the saved evidence first

A completed fit is identified by cohort, split, mask donor mapping, input windows, cache hashes, architecture, optimizer and seed. The reader reports missing or incompatible records without replacing them. Validation chooses the best epoch; test predictions are retained for later paired comparisons.


In [ ]:
inventory = pd.DataFrame(study.saved_inventory(
    RESULTS_ROOT, dataset_key=DATASET_KEY, seeds=TRAINING_SEEDS,
    include_optional=INCLUDE_OPTIONAL,
))
display(inventory.groupby(["rna_encoder", "grid", "status"], dropna=False).size().rename("fits").to_frame())
if (inventory["status"] == "incompatible").any():
    display(inventory.loc[inventory["status"] == "incompatible", ["path", "issue"]])


## Prepare one common transcript cohort

A source row supplies a transcript sequence, a codon-start track, a gene identifier and its target. Codon-start marks define the CDS in transcript coordinates. Translation must contain complete unambiguous codons, no internal stop and at least one residue; when a protein string is supplied it must match the translation. Ambiguity anywhere in the transcript, invalid coding annotations, missing gene IDs and non-finite targets are recorded as exclusion reasons. All methods use the same retained rows.

The human homology map is required before the 70/15/15 allocation. Mapped genes retain their homology groups; rows without a map entry are grouped by their recorded gene identifier. This preserves their eligibility while keeping repeated observations of the same gene together. It is our explicit adaptation of the [released mRNABench splitter](https://github.com/morrislab/mRNABench/blob/778d691ab91c51c03addeb9219229f2ec81d195c/mrna_bench/data_splitter/homology_split.py), which assigns unmapped rows individually. Gene aliases are not normalized.

This split separates recorded families and gene identities, but cannot guarantee that unrecorded homologs stay together. Exact sequence overlap is checked as well; any remaining overlap stops preparation for review. Group allocation gives approximate row proportions, not exact row quotas. No expression target is used to select the split or mask donor.

Preparation saves the ordered cohort identities, map hash, grouping policy, excluded-row reasons, split membership and a donor permutation inside each partition. Reuse requires the same identities and policy; a mismatch leaves the existing record intact. For a recipient of length $L_r$, donor endpoints $s_d,e_d$ from length $L_d$ map to $\lfloor s_dL_r/L_d\rfloor$ and $\lceil e_dL_r/L_d\rceil$. This is a negative control: it changes a mask, not the recipient’s protein sequence or its true residue availability. It preserves the cohort without restricting it to equal-length transcripts.


In [ ]:
needs_inputs = RUN_PREPARATION or RUN_EXTRACTION or RUN_TRAINING or RUN_DIAGNOSTICS or RUN_INPUT_AUDIT
cohort = preparation = plans = None
if needs_inputs:
    frame, source = region.load_local_frame(DATASET_KEY, DATA_CACHE_ROOT)
    homology, map_source = region.load_local_homology_map(DATA_CACHE_ROOT, "human")
    source = {**source, "homology_map_sha256": map_source["sha256"]}
    cohort = region.prepare_cohort(
        frame, DATASET_KEY, grouping="homology", homology_map=homology,
        unmapped_gene_policy=UNMAPPED_GENE_POLICY, source=source,
    )
    if RUN_PREPARATION or RUN_INPUT_AUDIT:
        # The audit examines this candidate split in memory; only preparation publishes it.
        preparation = study.prepare_study(cohort, split_seed=SPLIT_SEED, mask_seed=MASK_SEED)
        if RUN_PREPARATION:
            study.publish_preparation(PREPARATION_PATH, preparation)
            print(f"Prepared {len(cohort.frame)} eligible rows; excluded {len(cohort.exclusions)} rows.")
            display(pd.DataFrame(cohort.exclusions))
    else:
        preparation = study.read_preparation(PREPARATION_PATH, cohort)
    plans = region.plan_chunks(cohort, window_lengths=WINDOW_LENGTHS)
    display(pd.DataFrame([
        {
            "partition": name, "rows": len(rows),
            "evaluation groups": len(set(cohort.groups[rows])),
            "mapped rows": sum(cohort.groups[row].startswith("homology:") for row in rows),
            "unmapped rows": sum(cohort.groups[row].startswith("gene:") for row in rows),
        }
        for name, rows in preparation["positions"].items()
    ]))
else:
    print("Saved-evidence mode: source parquet, homology map and token caches were not opened.")


### Does the retained population support the comparison?

`RUN_INPUT_AUDIT` reads the local inputs, applies the agreed rules and inspects the fixed candidate split without opening embedding caches or training models. It saves one reusable audit JSON, separate from the canonical training preparation. A different audit identity is refused at an existing destination; preserve that evidence and explicitly choose a separate study destination when revising the protocol.

Reading mode loads that saved JSON without needing the original dataset. Its identity describes the inspected inputs; displaying it does not verify that today's files are unchanged. The plots below use saved measurements and make no automatic inclusion or split decisions. Target values are omitted from this input audit so test outcomes cannot become a design-selection tool.


In [ ]:
if RUN_INPUT_AUDIT:
    measured_inputs = region.input_audit(
        frame, cohort, plans, preparation, widths=AUDIT_WIDTHS, dtype=CACHE_DTYPE,
    )
    audit_producer = PRODUCER or run_identity.producer_record(device="cpu", repository=REPO_DIR)
    input_audit = region.publish_input_audit(INPUT_AUDIT_PATH, measured_inputs, producer=audit_producer)
else:
    input_audit = region.read_input_audit(INPUT_AUDIT_PATH)
if input_audit is None:
    print("No saved input audit. Enable RUN_INPUT_AUDIT for an explicitly recorded CPU inspection.")
else:
    print("Reading input audit:", INPUT_AUDIT_PATH)
    print("Inspected cohort:", input_audit["context"]["cohort_sha256"])
    audit_rows = pd.DataFrame(input_audit["rows"])
    audit_groups = pd.DataFrame(input_audit["groups"])
    display(audit_rows.groupby("status").size().rename("rows").to_frame())
    reasons = audit_rows.loc[audit_rows["status"] == "excluded", "reasons"].explode().dropna()
    display(reasons.value_counts().rename("excluded rows with reason").to_frame())
    display(pd.json_normalize(input_audit["context"]["preparation_policy"]))
    display(pd.DataFrame(input_audit["partitions"]))
    retained_rows = audit_rows.loc[audit_rows["status"] == "retained"]
    display(retained_rows.groupby("group_kind").size().rename("retained rows").to_frame())


In [ ]:
if input_audit is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    length_bins = np.histogram_bin_edges(audit_rows["length"].dropna(), bins=30)
    for status, rows in audit_rows.groupby("status"):
        axes[0].hist(rows["length"].dropna(), bins=length_bins, histtype="step", label=status)
    axes[0].set(xlabel="Transcript nucleotides", ylabel="Rows", title="Which lengths remain?")
    axes[0].legend()
    for kind, groups in audit_groups.groupby("group_kind"):
        axes[1].hist(groups["rows"], bins=20, histtype="step", label=kind)
    axes[1].set(xlabel="Rows per evaluation group", ylabel="Groups", title="How concentrated are the groups?")
    axes[1].legend()
    fig.tight_layout()
    plt.show()


Exclusion reasons can overlap: one row may appear under multiple reasons, so their counts must not be added as distinct exclusions. Exclusion differences would narrow the population our result describes; they do not authorize dropping additional rows or changing the agreed policy. Large groups can make row proportions diverge from the nominal split and reduce independent evaluation support. An unmapped-gene group still cannot rule out homology with another group. These are reasons to review the claim or protocol, not to search for a favorable split seed.

Known region lengths can also be inspected for excluded rows when their coding annotation remains valid. Missing region measurements are not zeros. For retained rows, absent or short UTRs limit how much of the comparison tests untranslated regulation rather than coding-region prediction. [Saluki's region and positional evidence](https://doi.org/10.1186/s13059-022-02811-x) motivates inspecting these inputs; it does not validate our fusion design.


In [ ]:
if input_audit is not None:
    display(retained_rows[["utr5", "cds", "utr3", "protein_length"]].describe())
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for name, label in (("utr5", "5-prime UTR"), ("cds", "CDS"), ("utr3", "3-prime UTR")):
        values = np.sort(retained_rows[name].dropna().to_numpy())
        ax.step(values, np.arange(1, len(values) + 1) / len(values), where="post", label=label)
    ax.set(xlabel="Region length in nucleotides", ylabel="Fraction at or below this length",
           title="What biological regions can the retained cohort test?")
    ax.legend()
    fig.tight_layout()
    plt.show()


---

## Preserve positions through encoder windows

RNA-FM and RiNALMo supply one feature vector per RNA base; ESM-2 supplies one per translated amino-acid residue. Independent, non-overlapping windows cover the complete transcript or translated CDS in order. The cache validates every window’s input and token count; reassembling them restores coordinates, **not context the encoder never saw across a window boundary**.

Each residue vector is repeated at its three coding bases. UTRs and the terminal stop receive zero protein features and a false protein-availability mask. A numerical zero does not itself mean a feature is unavailable. This distinction prevents a projection bias or attention operation from inventing a protein contribution at a position without a residue.

The pooled control averages CDS codons in their annotated frame and UTR triplets separately, retaining partial UTR-edge bins. No bin crosses a CDS/UTR boundary. A permuted mask on this coarse grid uses each bin’s midpoint membership; it does not move protein vectors. This explicit projection changes both resolution and the physical span of a TextCNN kernel.


In [ ]:
if plans is not None and not RUN_INPUT_AUDIT:
    display(pd.DataFrame(region.cache_inventory(
        plans, TOKEN_CACHE_ROOT, chunk_size=CACHE_SHARD_ROWS, dtype=CACHE_DTYPE,
    ))[["encoder", "chunks", "status", "issues"]])
    print("Metadata inventory is not shard verification; fitting verifies the selected caches.")


### What does the window choice cost in context and storage?

The table estimates token-array payload from retained positions, checkpoint feature widths and cache dtype. It excludes file overhead, model weights, temporary buffers and training memory. Window counts describe extraction calls; they do not reduce the full reassembled sequence length seen by the fusion head. Native and pooled grids also change a TextCNN kernel's span in nucleotides, so their score difference cannot isolate information loss alone.

Frequent multiwindow inputs would qualify any claim about long-range encoder context. A large payload or long-input tail motivates a bounded Newton resource trial, not an automatic crop. Runtime and peak memory require actual measurements before scheduling the panel.


In [ ]:
if input_audit is not None:
    window_summary = pd.DataFrame(input_audit["windows"])
    window_summary["payload GB"] = window_summary["payload_bytes"] / 1e9
    display(window_summary.drop(columns="payload_bytes"))
    display(retained_rows[["length", "pooled_positions", "rnafm_windows", "rinalmo_windows",
                           "esm2_8m_windows", "rnafm_boundaries_in_cds", "rinalmo_boundaries_in_cds"]].describe())


### Does the shuffled mask change the intended control?

A different donor transcript does not guarantee a different mask. Furthermore, D/E permit protein updates only where the supplied mask intersects true protein availability. A changed UTR annotation may therefore leave the effective update positions unchanged. Inspect both the annotation overlap and the fraction of real protein-update positions retained, separately on the native and pooled grids.

This control preserves protein correspondence and can remove permitted updates; it does not move residues into UTRs. If most effective masks remain unchanged, a null true-versus-permuted result would be weak evidence about region usefulness. If almost all updates disappear, the contrast also approaches protein removal. Both findings warrant review; neither causes automatic donor reselection or a new mask seed.


In [ ]:
if input_audit is not None:
    audit_masks = pd.DataFrame(input_audit["masks"])
    display(audit_masks.groupby("grid")[["mask_overlap", "retained_update_fraction",
                                         "effective_mask_unchanged"]].mean())
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for grid, rows in audit_masks.groupby("grid"):
        ax.hist(rows["retained_update_fraction"].dropna(), bins=np.linspace(0, 1, 21),
                histtype="step", label=grid)
    ax.set(xlabel="Fraction of protein-update positions retained after mask permutation",
           ylabel="Transcripts", title="Does the control meaningfully change available updates?")
    ax.legend()
    fig.tight_layout()
    plt.show()


In [ ]:
if RUN_EXTRACTION:
    # Only this branch may load pretrained weights and write new token-cache shards.
    with region.open_token_caches(
        plans, TOKEN_CACHE_ROOT, mode="extract", device=DEVICE,
        chunk_size=CACHE_SHARD_ROWS, dtype=CACHE_DTYPE,
    ):
        pass
    print("Requested token windows are complete; compatible existing shards were reused.")
else:
    print("Token extraction disabled.")


## What each comparison asks

| Arm | Inputs/operation | Question |
| --- | --- | --- |
| A | RNA only | What can a strong RNA representation predict alone? |
| B | RNA + aligned protein; zero protein outside actual coding residues; concatenation | Does the additional representation help? |
| C | B + an explicit region-mask feature | Does exposing the region annotation add predictive value? |
| D | C + independent position-wise sigmoid gates | Does adaptive local weighting improve C? |
| E | RNA queries protein; CDS-restricted gated updates and RNA skip | Does position-specific use of global protein attention improve D? |

C, D and E each have true- and transcript-permuted-mask variants. Protein keys and feature availability remain biologically valid in both. D/E protein updates require both the supplied control mask and true protein availability. E therefore returns its projected RNA skip exactly when protein updates are disabled. Each permitted RNA query can attend to every valid protein key; the CDS restriction does not create a codon-neighborhood attention window. No entropy penalty is added to these sigmoid gates. [Mina’s plan, Steps 4–6](../../docs/Biological_Fusion_Plan.docx)

All arms use a 100-channel TextCNN predictor after fusion. C/D use a concatenation mixer; E uses an additive RNA skip. Their parameter counts and operations differ, so E−D is a comparison of the specified systems rather than attention alone. The optional ungated-attention arm also allows updates at UTR queries; comparing it with E changes both gating and query availability. It is not a gate-only control. The optional nongated positional arm has a different capacity and bottleneck; its presence does not establish a matched control. No DNA stream is constructed.

The zero-filled protein stream already reveals some residue availability. C minus B tests whether exposing an annotation explicitly helps prediction; it does not establish that a wholly new biological input was introduced. The corrected D gates see RNA and protein jointly. D retains C’s concatenation/mask readout, whereas E adds attention updates to RNA; this readout difference remains explicit.


In [ ]:
display(pd.DataFrame([asdict(spec) for spec in PANEL]))


---

## Fit missing cells or inspect completed weights

Training opens only the selected RNA encoder and, where needed, protein tokens. Each arm/grid/encoder/seed has its own destination. The shared trainer retains best-validation weights, per-row test predictions and epoch-boundary recovery state. An incompatible completed record is refused, not overwritten. Selecting a new seed does not change the cohort, split or control permutation.

Diagnostics are separate inference from checksum-verified best weights. They retain per-transcript mean gate/update values for the true 5′UTR, CDS and 3′UTR. Absent weights never trigger retraining. Gate patterns describe the fitted computation; they do not establish biological importance without perturbation evidence.

Norm diagnostics accompany gate weights when available: a low scalar weight can multiply a large feature vector. They describe contribution magnitude, not causal or biological importance. Missing norm evidence in an older diagnostic record stays missing until separately authorized inference; it is not reconstructed during reading.


In [ ]:
if RUN_TRAINING or RUN_DIAGNOSTICS:
    for spec in PANEL:
        requested = [spec.rna_encoder]
        if spec.method != "rna_only":
            requested.append("esm2_8m")
        selected_plans = {key: plans[key] for key in requested}
        with region.open_token_caches(
            selected_plans, TOKEN_CACHE_ROOT, mode="read",
            chunk_size=CACHE_SHARD_ROWS, dtype=CACHE_DTYPE,
        ) as caches:
            streams = study.RegionStreams(cohort, caches, preparation, spec)
            if RUN_TRAINING:
                result = study.run_fit(
                    streams, CONFIG, MODEL_SETTINGS, STUDY_ROOT / "fits", DEVICE,
                    producer=PRODUCER,
                )
                print(f"{spec.key}: seed {SEED}, test Spearman {result.test_score:.4f}")
            else:
                study.collect_diagnostics(
                    streams, CONFIG, MODEL_SETTINGS, STUDY_ROOT / "fits", DEVICE,
                    producer=PRODUCER,
                )
else:
    print("Training and best-weight diagnostics disabled.")


### Read contributions beside the gates

The reader below opens saved diagnostic and fit JSON only. New v2 records include norms; legacy v1 records remain labeled and do not acquire missing measurements. A scalar gate cannot describe contribution magnitude by itself: its vector may be large, or its update may be disabled by availability. Compare true biological regions even for the permuted-mask arm.

The plot uses the stored **mean post-gate update norm divided by mean anchor norm** within each transcript's CDS. This is a ratio of regional means, not a mean of position-wise ratios. It measures numerical scale, not causal importance or molecular regulation. The pre-gate update already includes the selected normalization. No diagnostic is generated during reading.


In [ ]:
diagnostic_status, diagnostic_values = [], []
for spec in PANEL:
    for seed in TRAINING_SEEDS:
        path = STUDY_ROOT / "fits" / spec.key / f"seed{seed}.json"
        saved = study.read_diagnostics(path)
        if saved is None:
            continue
        diagnostic_status.append({"fit": spec.key, "seed": seed,
                                  "version": saved["version"], "norms": saved["norm_status"],
                                  "integrity": saved["integrity"]})
        for row in saved["record"]["rows"]:
            for name, values in row["regions"].items():
                diagnostic_values.append({"fit": spec.key, "seed": seed, "row_id": row["row_id"],
                                          "region": name, **values})
if diagnostic_status:
    display(pd.DataFrame(diagnostic_status))
    region_diagnostics = pd.DataFrame(diagnostic_values)
    display(region_diagnostics.head(12))
else:
    print("No saved region diagnostics. Reading did not run inference.")


In [ ]:
if diagnostic_values and {"anchor_norm", "update_norms"} <= set(region_diagnostics.columns):
    scale_rows = region_diagnostics.loc[
        (region_diagnostics["region"] == "cds") & region_diagnostics["anchor_norm"].notna()
    ]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for fit, rows in scale_rows.groupby("fit"):
        ratios = [float(np.asarray(update).sum()) / anchor
                  for update, anchor in zip(rows["update_norms"], rows["anchor_norm"])
                  if update is not None and anchor > 0]
        if ratios:
            values = np.sort(ratios)
            ax.step(values, np.arange(1, len(values) + 1) / len(values), where="post", label=fit)
    ax.set(xlabel="CDS mean update norm / CDS mean anchor norm",
           ylabel="Fraction of transcript-seed observations", title="How large are admitted protein updates?")
    if ax.lines:
        ax.legend(fontsize=7, bbox_to_anchor=(1, 1), loc="upper left")
    fig.tight_layout()
    plt.show()


## Compare the complete three-seed evidence

The report requires all core arms and all three training seeds for this dataset. It calculates B−A, C−B, D−C and E−D; true−permuted masks; and native−pooled RNA. Each paired bootstrap draw resamples whole recorded test groups (mapped homology groups or unmapped-gene groups) and averages the three **per-seed scores**. This is not ensembling the predictions. The seed standard deviation is reported separately from the interval over held-out groups.

`RUN_ANALYSIS` authorizes those calculations from saved predictions; it does not load caches or train. Saved reports bind the fit hashes, groups, ordered test rows, methods and seeds. Undefined statistics stop publication. A missing report remains missing until this explicit step is run.


In [ ]:
report = None
if (STUDY_ROOT / "preparation.json").is_file():
    try:
        report = study.saved_comparisons(
            RESULTS_ROOT, dataset_key=DATASET_KEY, draws=BOOTSTRAP_DRAWS,
            bootstrap_seed=BOOTSTRAP_SEED, producer=PRODUCER, run_analysis=RUN_ANALYSIS,
        )
    except (FileNotFoundError, ValueError) as error:
        if RUN_ANALYSIS:
            raise
        print(f"Three-seed report unavailable: {error}")
if report is not None:
    display(pd.DataFrame(report["summary"]))
    display(pd.DataFrame(report["contrasts"]))
else:
    print("No compatible completed three-seed report is available.")


In [ ]:
if report is not None:
    # Plot stored contrasts; do not rerun a fit, metric or bootstrap to redraw them.
    differences = pd.DataFrame(report["contrasts"])
    selected = differences[differences["comparison"].str.contains(r"/(?:B-A|C-B|D-C|E-D)$", regex=True)]
    fig, ax = plt.subplots(figsize=(8, max(4, len(selected) * .28)))
    positions = np.arange(len(selected))
    ax.hlines(positions, selected["ci_low"], selected["ci_high"], color="tab:blue")
    ax.plot(selected["difference"], positions, "o", color="tab:blue")
    ax.axvline(0, color="gray", linewidth=1)
    ax.set_yticks(positions, selected["comparison"])
    ax.set_xlabel("Spearman difference (first minus second); paired group interval")
    ax.invert_yaxis()
    fig.tight_layout()
    plt.show()


---

## What the eventual result would establish

A positive C−B would support using region annotations in this specified predictor. A positive D−C would motivate adaptive local weighting, while still requiring the capacity qualification. E−D compares gated attention with the local-gate system; it cannot isolate attention from its different skip/mixing structure. True masks outperforming permuted masks would make the annotation’s placement relevant to prediction, without proving a molecular mechanism.

Native/pooled differences also involve a different effective receptive field. Chunked embeddings omit interactions across encoder windows, even though no transcript positions are discarded. Held-out performance tests separation under this recorded map and gene identifiers, not complete homology separation or transfer to unseen species or experimental conditions. Half-life is the first investigation; the same protocol on TE asks whether any pattern transfers to a different biological target.

All neural arms retain a common BLF optimization policy. Their scores would compare these systems under that policy, not establish each architecture's best achievable performance.

The [Stage 4 investigation](README.md) owns the wider synthesis. These prepared methods do not constitute executed findings; resource profiling, complete seed coverage and the saved comparisons must precede a conclusion.

Before a Newton run, review the inspected population and control strength, model/control capacity and a measured resource trial. After fits, interpret the prespecified contrasts and seed variation before proposing another architecture. Retain, revise or defer a choice through explicit review; none of the supporting plots automatically changes the experiment.
